# Notebook for merging deidentified "book a librarian" bioethics export data

In [23]:
import pandas as pd
import plotly.express as px
import nbformat

In [24]:
# create dataframes for book a librarian datasets
df_bioethics = pd.read_excel('/home/ekmys/PycharmProjects/Your-Library-Your-Impact/dashboard/data/book a librarian_bioethics/BAL_bioethics_all.xlsx')

In [46]:
def load_bookings_bioethics() -> pd.DataFrame:
    # read in excel file
    df_bioethics = pd.read_excel('/home/ekmys/PycharmProjects/Your-Library-Your-Impact/dashboard/data/book a librarian_bioethics/BAL_bioethics_all.xlsx')

    # Parse dates so we can sort and group by time
    df_bioethics["Date"] = pd.to_datetime(df_bioethics["Date"], errors="coerce")
    df_bioethics = df_bioethics.dropna(subset=["Date"])

    # Add a Year-Quarter label for grouping in charts
    df_bioethics["YearQuarter"] = (
    df_bioethics["Date"].dt.year.astype(str)
    + " Q"
    + df_bioethics["Date"].dt.quarter.astype(str))

    # rename columns
    df_bioethics = df_bioethics.rename(columns={" Custom Fields (Topics)": "Bioethics Topic"})

    # remove whitespace at head or tail
    df_bioethics['Bioethics Topic'] = df_bioethics['Bioethics Topic'].str.strip()

    # consolidate multiple-category topics
    df_bioethics['Bioethics Topic'] = df_bioethics['Bioethics Topic'].replace('Physician Aid-in-Dying AND Do Not Resuscitate Orders AND End-of-Life Issues AND Parental Decision Making','Physician Aid in Dying')
    df_bioethics['Bioethics Topic'] = df_bioethics['Bioethics Topic'].replace('End-of-Life Issues AND (Treatment Refusal OR Cross-Cultural Issues and Diverse Beliefs)', 'End-of-Life Issues')
    df_bioethics['Bioethics Topic'] = df_bioethics['Bioethics Topic'].replace('End-of-Life Issues OR Termination of Life-Sustaining Treatment', 'End-of-Life Issues')
    df_bioethics['Bioethics Topic'] = df_bioethics['Bioethics Topic'].replace('Public Health Ethics AND Mistakes', 'Public Health Ethics')

    return df_bioethics

In [47]:
df_bioethics = load_bookings_bioethics()

In [48]:
df_bioethics.head()

,Date,Service,Location,Duration (mins.),Signed Up Attendees Count,Bioethics Topic,Academic Year,Quarter,YearQuarter
0,2023-07-03,ELEC 704 Bioethics,Virtual,60,1,Student Issues,2023-2024,Q1,2023 Q3
1,2023-07-06,ELEC 704 Bioethics,Virtual,60,1,Public Health Ethics,2023-2024,Q1,2023 Q3
2,2023-07-12,ELEC 704 Bioethics,Virtual,60,1,Truth-telling and Withholding Information,2023-2024,Q1,2023 Q3
3,2023-07-12,ELEC 704 Bioethics,Virtual,60,1,Unsure,2023-2024,Q1,2023 Q3
4,2023-07-14,ELEC 704 Bioethics,Virtual,60,1,Unsure,2023-2024,Q1,2023 Q3


In [49]:
df_bioethics.info()

<class 'pandas.DataFrame'>
RangeIndex: 76 entries, 0 to 75
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Date                       76 non-null     datetime64[us]
 1   Service                    76 non-null     str           
 2   Location                   73 non-null     str           
 3   Duration (mins.)           76 non-null     int64         
 4   Signed Up Attendees Count  76 non-null     int64         
 5   Bioethics Topic            76 non-null     str           
 6   Academic Year              76 non-null     str           
 7   Quarter                    76 non-null     str           
 8   YearQuarter                76 non-null     str           
dtypes: datetime64[us](1), int64(2), str(6)
memory usage: 10.7 KB


In [29]:
df_bioethics = df_bioethics.rename(columns={" Custom Fields (Topics)": "Bioethics Topic"})

In [38]:
# remove whitespace at head or tail
df_bioethics['Bioethics Topic'] = df_bioethics['Bioethics Topic'].str.strip()

In [39]:
unique_topics = list(df_bioethics['Bioethics Topic'].unique())
print(unique_topics)

['Student Issues', 'Public Health Ethics', 'Truth-telling and Withholding Information', 'Unsure', 'Maternal / Fetal Conflict', 'Parental Decision Making', 'Physician Aid-in-Dying AND Do Not Resuscitate Orders AND End-of-Life Issues AND Parental Decision Making', 'End-of-Life Issues AND (Treatment Refusal OR Cross-Cultural Issues and Diverse Beliefs)', 'Treatment Refusal', 'Clinical Ethics and Law', 'Physician-Patient Relationship', 'Complementary Medicine', 'Ethics Committees and Consultation', 'Physician Aid-in-Dying', 'End-of-Life Issues OR Termination of Life-Sustaining Treatment', 'End-of-Life Issues', 'Informed Consent', 'Resource Allocation', 'Spirituality and Medicine', 'Research Ethics', 'Public Health Ethics AND Mistakes', 'Genetics']


In [40]:
# for anything in 'Bioethics Topics' labeled 'End of Life Issues' just call it 'End of Life Issues'
df_bioethics['Bioethics Topic'] = df_bioethics['Bioethics Topic'].replace('Physician Aid-in-Dying AND Do Not Resuscitate Orders AND End-of-Life Issues AND Parental Decision Making','Physician Aid in Dying')
df_bioethics['Bioethics Topic'] = df_bioethics['Bioethics Topic'].replace('End-of-Life Issues AND (Treatment Refusal OR Cross-Cultural Issues and Diverse Beliefs)', 'End-of-Life Issues')
df_bioethics['Bioethics Topic'] = df_bioethics['Bioethics Topic'].replace('End-of-Life Issues OR Termination of Life-Sustaining Treatment', 'End-of-Life Issues')
df_bioethics['Bioethics Topic'] = df_bioethics['Bioethics Topic'].replace('Public Health Ethics AND Mistakes', 'Public Health Ethics')

In [41]:
unique_topics = list(df_bioethics['Bioethics Topic'].unique())
print(unique_topics)

['Student Issues', 'Public Health Ethics', 'Truth-telling and Withholding Information', 'Unsure', 'Maternal / Fetal Conflict', 'Parental Decision Making', 'Physician Aid in Dying', 'End-of-Life Issues', 'Treatment Refusal', 'Clinical Ethics and Law', 'Physician-Patient Relationship', 'Complementary Medicine', 'Ethics Committees and Consultation', 'Physician Aid-in-Dying', 'Informed Consent', 'Resource Allocation', 'Spirituality and Medicine', 'Research Ethics', 'Genetics']


In [42]:
# create groupby object
agg = (
    df_bioethics.groupby("Bioethics Topic", as_index=False)
    .size()
)

In [43]:
agg.head()

,Bioethics Topic,size
0,Clinical Ethics and Law,7
1,Complementary Medicine,1
2,End-of-Life Issues,9
3,Ethics Committees and Consultation,1
4,Genetics,1


In [45]:
# PLOTLY BARPLOT VERSION
labels = {'Bioethics Topic':'Bioethics Topic', 'size':'Number of Bookings'}
fig = px.bar(agg, x=agg['Bioethics Topic'], y=agg['size'], labels=labels, title='Number of Bookings by Bioethics Topic')
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
def plot_bioethics_topics(df: pd.DataFrame) -> None:
    """Plots bioethics topics discussed in Book A Librarian appointments.
    """
    if df.empty:
        st.info("No data available.")
        return

    # create groupby object
    agg = (df_bioethics.groupby("Bioethics Topic", as_index=False).size())

    # PLOTLY BARPLOT VERSION
    labels = {'Bioethics Topic':'Bioethics Topic', 'size':'Number of Bookings'}
    fig = px.bar(agg, x=agg['Bioethics Topic'], y=agg['size'], labels=labels, title='Number of Bookings by Bioethics Topic')
    fig.update_xaxes(tickangle=45)
    st.plotly_chart(fig, use_container_width=True)